In [ ]:
!pip install xgboost flask ngrok pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 37.0 MB/s eta 0:00:00


In [ ]:
zip_path = "archive (2).zip"
extract_path = "data"

import zipfile

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted successfully!")

Extracted successfully!


In [ ]:
import os

os.listdir("/content/data")

['WA_Fn-UseC_-Telco-Customer-Churn.csv']

In [ ]:
import pandas as pd

df = pd.read_csv("/content/data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
# Drop ID column
df.drop('customerID', axis=1, inplace=True)

# Convert target
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Fix TotalCharges (common issue)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Remove nulls
df = df.dropna()

In [ ]:
X = df[['tenure', 'MonthlyCharges', 'TotalCharges']]
y = df['Churn']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier()
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.7697228144989339


In [ ]:
import pickle

pickle.dump(model, open('/content/churn_model.pkl', 'wb'))

In [ ]:
%%writefile app.py
from flask import Flask, render_template, request
import pickle
import numpy as np

app = Flask(__name__)

model = pickle.load(open('churn_model.pkl', 'rb'))

@app.route('/')
def home():
    return render_template('index.html')

@app.route('/predict', methods=['POST'])
def predict():
    tenure = float(request.form['tenure'])
    monthly = float(request.form['monthly'])
    total = float(request.form['total'])

    features = [[tenure, monthly, total]]

    prediction = model.predict(features)[0]

    result = "Will Churn" if prediction == 1 else "Will Stay"

    return render_template('index.html', prediction_text=result)

if __name__ == "__main__":
    app.run()

Writing app.py


In [ ]:
import os
os.makedirs("templates", exist_ok=True)

In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html>
<head>
    <title>Churn Prediction</title>
</head>
<body>

<h2>Customer Churn Prediction</h2>

<form action="/predict" method="post">
    Tenure: <input type="text" name="tenure"><br><br>
    Monthly Charges: <input type="text" name="monthly"><br><br>
    Total Charges: <input type="text" name="total"><br><br>

    <button type="submit">Predict</button>
</form>

<h3>{{ prediction_text }}</h3>

</body>
</html>

Writing templates/index.html


In [ ]:
!pip install pyngrok

In [ ]:
from pyngrok import ngrok

!python app.py &

public_url = ngrok.connect(5000)
print(public_url)

 * Serving Flask app 'app'
 * Debug mode: off
 * Running on http://127.0.0.1:5000
Press CTRL+C to quit


KeyboardInterrupt: 

In [ ]:
!nohup python app.py > log.txt 2>&1 &

In [ ]:
from pyngrok import ngrok
url = ngrok.connect(5000)
print(url)

ERROR:pyngrok.process.ngrok:t=2026-04-05T09:58:28+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-05T09:58:28+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-05T09:58:28+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
import pickle

# Load model
model = pickle.load(open('churn_model.pkl', 'rb'))

# Prediction function
def predict(tenure, monthly, total):
    prediction = model.predict([[tenure, monthly, total]])[0]
    prob = model.predict_proba([[tenure, monthly, total]])[0][1]

    if prediction == 1:
        return f"⚠️ Customer Will Churn\nProbability: {prob*100:.2f}%"
    else:
        return f"✅ Customer Will Stay\nProbability: {(1-prob)*100:.2f}%"

# Create UI
interface = gr.Interface(
    fn=predict,
    inputs=[
        gr.Slider(0, 72, label="Tenure (months)"),
        gr.Number(label="Monthly Charges"),
        gr.Number(label="Total Charges")
    ],
    outputs=gr.Textbox(label="Prediction Result"),
    title="💡 Customer Churn Predictor",
    description="Simple ML app using XGBoost"
)

# Launch app
interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d82027daf8bd58a031.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [ ]:
y_pred = model.predict(X_test)

NameError: name 'X_test' is not defined